# TESDA-TRACK — what learners say about their recommendations

Upload `feedback.csv`. Make it on the project's computer, from `backend/`, with `DATABASE_URL` pointing at the
database the answers are in (the pilot server's, not a laptop's test data):

```
../.venv/Scripts/python.exe -m training.feedback --output training/runs/feedback.csv
```

No GPU is needed. Run all cells; the last one downloads the worklist.

Each row is one answer to **"Are you satisfied with this recommendation?"** with the numbers behind that
recommendation. The goal's words were never kept, so these answers **can't fine-tune the matcher directly**. They show
where matching, the readiness result or the pathway disappointed learners, and which qualifications the next training
round should write examples for. The admin Reports page shows the same totals; this adds ranges, the score and
readiness checks, and the worklist.

The file has no goal text, account or learner details, but it is pilot data: keep it within the pilot team and don't
publish it, for example as a public Hugging Face dataset.

An answer is one click, and a few answers say little. Every rate comes with its count and a 95% range (Wilson
interval): read the range, not just the percentage.

## 1. Settings

In [ ]:
MIN_ANSWERS = 5  # a qualification needs this many answers before it can be flagged
TARGET = 0.8     # flag it when even the top of its 95% range is below this satisfaction rate

## 2. Upload and check the export

In [ ]:
import io
import math

import pandas as pd
from google.colab import files

COLUMNS = ["month", "qualification_code", "qualification_name", "place", "rank", "match_count", "score", "keyword",
           "z_score", "model", "z_floor", "z_ceiling", "keyword_weight", "min_score", "intent", "readiness_score",
           "readiness_level", "satisfied", "reason", "wanted_code", "wanted_name"]
uploaded = files.upload()
assert len(uploaded) == 1, "Upload exactly one feedback.csv"
df = pd.read_csv(io.BytesIO(next(iter(uploaded.values()))), keep_default_na=False, na_values=[""])
assert list(df.columns) == COLUMNS, "Not an export from this version of training.feedback: export it again"
assert len(df), "The export has no answers yet"
assert df["satisfied"].dtype == bool, "The satisfied column must hold only True and False"
df["model"] = df["model"].fillna("keyword only")
df["reason"] = df["reason"].fillna("")
UNSATISFIED = ~df["satisfied"]
print(f"{len(df)} answers from {df.month.min()} to {df.month.max()}: {df.satisfied.sum()} satisfied "
      f"({df.satisfied.mean():.0%})")
if len(df) < 100:
    print("Fewer than 100 answers: read the counts and ranges, not just the percentages.")


def wilson(k, n, z=1.96):
    """The 95% range for k satisfied answers out of n."""
    p = k / n
    centre = (p + z * z / (2 * n)) / (1 + z * z / n)
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return round(centre - half, 3), round(centre + half, 3)


def satisfaction(frame, by):
    """Answers, satisfied answers, the rate and its 95% range, per group."""
    table = frame.groupby(by, observed=True).agg(answers=("satisfied", "size"), satisfied=("satisfied", "sum"))
    table["rate"] = (table.satisfied / table.answers).round(3)
    ranges = [wilson(k, n) for k, n in zip(table.satisfied, table.answers)]
    table["low"] = [low for low, _ in ranges]
    table["high"] = [high for _, high in ranges]
    return table

## 3. Overall
**model** is the matching model the answer was given with, so a new model's answers can be compared with the old
one's. **place**: `top_match` is the first match shown, `other_match` the second or third, and `own_choice` a
qualification the learner picked from the library instead of the matches.

In [ ]:
display(satisfaction(df, "model"))
display(satisfaction(df, "place"))
display(df[UNSATISFIED].reason.replace("", "no reason given").value_counts().rename("answers of No"))

## 4. What learners wanted instead
After "It's the wrong qualification for my goal", a learner can say which one they wanted. Each pair is a mix-up
matching made: write examples that tell the two apart, and new dev goals for both.

In [ ]:
instead = df[df.wanted_code.notna()]
if instead.empty:
    print("No learner has said which qualification they wanted yet.")
else:
    display(instead.groupby(["qualification_name", "wanted_name"]).size().rename("answers")
            .sort_values(ascending=False).to_frame())

## 5. Does the match score predict satisfaction?
Matches only (not library picks), by the score band the learner saw. If the low bands are mostly "wrong
qualification", the minimum score may be too low. The second table shows, for each model and its parameters, what a
higher minimum score would have hidden, listing a minimum only where it starts hiding more answers. Only the chosen
qualification's score is kept, so it can't show what the
learner would have seen instead: try a new minimum on the dev goals (`training.evaluate --params`) before changing
it.

In [ ]:
shown = df[df.place != "own_choice"].copy()
rows = []
if shown.empty:
    print("No answers about a shown match yet.")
else:
    shown["score_band"] = pd.cut(shown.score, [0, 40, 60, 80, 101], right=False,
                                 labels=["0-39", "40-59", "60-79", "80-100"])
    display(satisfaction(shown, ["model", "score_band"]))
    keys = ["model", "z_floor", "z_ceiling", "keyword_weight", "min_score"]
    for key, group in shown.dropna(subset=["min_score"]).groupby(keys):
        wrong = group[group.reason == "wrong_qualification"]
        happy = group[group.satisfied]
        last = None
        for minimum in range(int(key[-1]), 101, 5):
            hides = (f"{(wrong.score < minimum).sum()} of {len(wrong)}", f"{(happy.score < minimum).sum()} of {len(happy)}")
            if hides != last:  # list a minimum only where it starts hiding more
                rows.append(dict(zip(keys, key), new_min_score=minimum, hides_wrong=hides[0], hides_satisfied=hides[1]))
                last = hides
            if (group.score < minimum).all():
                break
    if rows:
        display(pd.DataFrame(rows).set_index(keys))
    else:
        print("Keyword-only answers keep no parameters to check.")

## 6. Does the readiness result fit?
Readiness comes from the learner's own ratings of each competency: 80 and above is High (consider assessment), 50 to
79 Moderate (focused training first), below 50 Low (formal training first). If "the readiness result doesn't seem
right" answers gather just below a boundary, the boundary or that qualification's competency questions may be off.

In [ ]:
display(satisfaction(df, "readiness_level"))
bands = pd.cut(df.readiness_score, [0, 40, 50, 60, 70, 80, 90, 101], right=False).rename("readiness_score")
display(df.assign(readiness_off=df.reason == "readiness_off")
        .groupby(bands, observed=True).agg(answers=("satisfied", "size"), readiness_off=("readiness_off", "sum")))

## 7. What to improve, per qualification
One row per qualification with an answer of No or a pick from the library. **improve** names its most frequent
problem:

- `matching`: "wrong qualification" answers, plus library picks (the learner had to find it themselves). Write new
  goal-style examples for it (`backend/training/data/synthetic/` or the v4 regional file) and new dev and sealed
  goals, rebuild the bundle, and train a new round.
- `readiness`: review its competency questions and the readiness levels.
- `pathway`: review its pathway steps and its training and assessment options.

**wanted_instead** lists what learners wanted when this one was wrong. **below_target** is True when it has at least
`MIN_ANSWERS` answers and even the top of its 95% range is under `TARGET`; those come first.

Goals that learners chose to share are not in this file: `python -m training.shared_goals collect` gathers them for
review (see `backend/training/README.md`).

In [ ]:
names = ["qualification_code", "qualification_name"]
table = satisfaction(df, names)


def count(mask):
    return df[mask].groupby(names).size().reindex(table.index, fill_value=0)


library = df.place == "own_choice"
table["wrong_qualification"] = count(df.reason == "wrong_qualification")
table["readiness_off"] = count(df.reason == "readiness_off")
table["pathway_not_helpful"] = count(df.reason == "pathway_not_helpful")
table["no_reason"] = count(UNSATISFIED & (df.reason == ""))
table["picked_from_library"] = count(library)
table["wanted_instead"] = (instead.groupby(names).wanted_name
                           .agg(lambda wanted: "; ".join(f"{name} ({n})" for name, n in wanted.value_counts().items()))
                           .reindex(table.index, fill_value=""))
problems = pd.DataFrame({"matching": table.wrong_qualification + count(library & (df.reason != "wrong_qualification")),
                         "readiness": table.readiness_off, "pathway": table.pathway_not_helpful})
table["improve"] = problems.idxmax(axis=1).where(problems.max(axis=1) > 0, "")
table["below_target"] = (table.answers >= MIN_ANSWERS) & (table.high < TARGET)
worklist = (table[(table.answers > table.satisfied) | (table.picked_from_library > 0)]
            .assign(unsatisfied=lambda t: t.answers - t.satisfied)
            .sort_values(["below_target", "unsatisfied", "answers"], ascending=False))
print(f"{len(worklist)} of {len(table)} qualifications with answers need a look; "
      f"{int(worklist.below_target.sum())} are below target.")
display(worklist)

## 8. Download the worklist
`feedback-worklist.csv` holds the table above. Fix `readiness` and `pathway` rows in the app's content. Only
`matching` rows go into the next training round, with newly written examples: the answers hold no goal text to copy.

In [ ]:
worklist.reset_index().to_csv("feedback-worklist.csv", index=False)
files.download("feedback-worklist.csv")